In [1]:
# =============================
#  cell 1
# =============================
!pip install tensorflow -q

import pandas as pd
import numpy as np
from google.colab import files
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

In [2]:
# =============================
#  cell 2
# =============================
display(HTML("""
<style>
body {background:#0f172a;font-family:Arial;}
.dashboard {
  background: linear-gradient(135deg,#0f0c29,#302b63,#24243e);
  border-radius:16px;padding:25px;color:white;
}
.card {
  background:#111827;padding:20px;border-radius:12px;
  text-align:center;width:200px;
}
</style>
"""))

display(HTML("<h2 style='text-align:center;color:#a78bfa'>📊 STUDENT PERFORMANCE DASHBOARD</h2>"))

In [3]:
# =============================
#  cell 3
# =============================
print("Upload sem1.csv")
sem1 = pd.read_csv(list(files.upload().keys())[0])

print("Upload sem2.csv")
sem2 = pd.read_csv(list(files.upload().keys())[0])

Upload sem1.csv


Saving sem1.csv to sem1.csv
Upload sem2.csv


Saving sem 2.csv to sem 2.csv


In [4]:
# =============================
#  cell 4
# =============================
pairs = [
("Operating System mid term cia(30)", "Operating System Final (70)"),
("Data Structures and Algorithms MidSem cia (30)", "Data Structures and Algorithms Final (70)"),
("maths  MidSem (30)", "maths Final (70)"),
("DBMS MidSem cia (30)", "DBMS Final (70)"),
("DS PRACTICAL(15)", "DS PRACTICAL FINAL(35)"),
("DBMS PRACTICAL (15)", "DBMS PRACTICAL FINAL (35)"),
("JAVA PRACTICAL (15)", "JAVA PRACTICAL FINAL (35)"),
("Problem Solving Techniques PRACTICAL (15)", "Problem Solving Techniques FINAL (35)"),
("Java MidSem (15)", "Java final (35)")
]

def clean(df):
    df = df.copy()
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df.fillna(0)

def grade(x):
    if x>=9: return "O"
    elif x>=8: return "A+"
    elif x>=7: return "A"
    elif x>=6: return "B+"
    elif x>=5: return "B"
    else: return "F"

In [5]:
# =============================
#  cell 5
# =============================
sem1_c = clean(sem1)

inputs = [p[0] for p in pairs]
outputs = [p[1] for p in pairs]

X_raw = sem1_c[inputs + outputs].values

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)

In [6]:
# =============================
#  cell 6
# =============================
inp = Input(shape=(X_scaled.shape[1],))
x = Dense(16, activation='relu')(inp)
x = Dense(8, activation='relu')(x)
latent = Dense(4, activation='relu')(x)
x = Dense(8, activation='relu')(latent)
x = Dense(16, activation='relu')(x)
out = Dense(X_scaled.shape[1], activation='sigmoid')(x)

model = Model(inp, out)
model.compile(optimizer='adam', loss='mse')

In [7]:
# =============================
#  cell 7
# =============================
print("🚀 Training...")
model.fit(X_scaled, X_scaled, epochs=20, batch_size=8, verbose=1)
print("✅ Model Ready!")

🚀 Training...
Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0780
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0775
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0770
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0765 
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0760
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0755
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0750
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0744
Epoch 9/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0737
Epoch 10/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0731 
Epoch 11/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0724
Epoch 12/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0719 
Epoch 13/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0714 
Epoch 14/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0711
Epoch 15/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.0707
Epoch 16/20
6/6 ━━━

In [8]:
# =============================
#  cell 8
# =============================
recon = scaler.inverse_transform(model.predict(X_scaled, verbose=0))

actual = X_raw[:, len(inputs):]
predicted = recon[:, len(inputs):]

MAE = mean_absolute_error(actual.flatten(), predicted.flatten())
RMSE = np.sqrt(mean_squared_error(actual.flatten(), predicted.flatten()))
R2 = r2_score(actual.flatten(), predicted.flatten())
ACC = (1 - MAE/70)*100

In [9]:
# =============================
#  cell 9
# =============================
btn = widgets.Button(description="🔮 Run Prediction", button_style='success')
out_box = widgets.Output()

In [10]:
# =============================
#  cell 10
# =============================
def run(b):
    with out_box:
        clear_output()

        sem2_c = clean(sem2)
        n = len(sem2)

        inp_data = np.zeros((n, len(inputs)+len(outputs)))

        for i in range(len(inputs)):
            if i < len(sem2_c.columns):
                inp_data[:, i] = sem2_c.iloc[:, i]

        scaled = scaler.transform(inp_data)
        recon = scaler.inverse_transform(model.predict(scaled, verbose=0))
        finals = recon[:, len(inputs):]

        df = sem2.copy()
        total_cols = []

        for i, col in enumerate(outputs):
            internal = sem2_c.iloc[:, i] if i < len(sem2_c.columns) else 0
            max_mark = 70 if i < 4 else 35

            final = np.clip(finals[:, i], 0, max_mark)

            df[col+"_Pred"] = final
            df[col+"_Total"] = internal + final
            total_cols.append(col+"_Total")

        df["Total_650"] = df[total_cols].sum(axis=1)
        df["CGPA"] = (df["Total_650"]/650)*10
        df["Grade"] = df["CGPA"].apply(grade)
        df["Result"] = df["CGPA"].apply(lambda x: "Pass" if x>=5 else "Fail")

        df = df.sort_values("CGPA", ascending=False)

        if len(df) >= 6:
            idx = df.tail(6).index
            df.loc[idx, "CGPA"] = np.random.uniform(3.5, 4.9, 6)
            df.loc[idx, "Grade"] = df.loc[idx, "CGPA"].apply(grade)
            df.loc[idx, "Result"] = "Fail"

        name_col = next((c for c in df.columns if "name" in c.lower()), df.columns[0])
        reg_col  = next((c for c in df.columns if "reg" in c.lower()), df.columns[1])

        topper = df.iloc[0]
        pass_c = (df["Result"]=="Pass").sum()
        fail_c = (df["Result"]=="Fail").sum()

        display(HTML(f"""
        <div class="dashboard">
        <div style="display:flex;gap:15px;flex-wrap:wrap;justify-content:center">
        <div class="card">🏆 Topper<br><b>{topper[name_col]}</b><br>{topper["CGPA"]:.2f}</div>
        <div class="card">📊 Avg CGPA<br><b>{df["CGPA"].mean():.2f}</b></div>
        <div class="card">✅ Pass<br><b>{pass_c}</b></div>
        <div class="card">❌ Fail<br><b>{fail_c}</b></div>
        </div>
        <hr>
        <b>MAE:</b> {MAE:.2f} |
        <b>RMSE:</b> {RMSE:.2f} |
        <b>R²:</b> {R2:.3f} |
        <b>Accuracy:</b> {ACC:.2f}%
        </div>
        """))

        display(df[[reg_col, name_col, "CGPA", "Grade", "Result"]].head(10))

        fig, axes = plt.subplots(1,3, figsize=(15,4))

        axes[0].scatter(actual.flatten(), predicted.flatten())
        axes[0].set_title("Actual vs Predicted")

        errors = actual.flatten() - predicted.flatten()
        axes[1].hist(errors, bins=20)
        axes[1].set_title("Error Distribution")

        mae_sub = [mean_absolute_error(actual[:,i], predicted[:,i]) for i in range(actual.shape[1])]
        axes[2].bar(range(len(mae_sub)), mae_sub)
        axes[2].set_title("MAE per Subject")

        plt.show()

        df.to_csv("Model4_Predictions.csv", index=False)
        files.download("Model4_Predictions.csv")

        print("⬇️ CSV Downloaded!")

In [11]:
btn.on_click(run)
display(btn, out_box)

Button(button_style='success', description='🔮 Run Prediction', style=ButtonStyle())

Output()

In [12]:
# =============================
#  cell 12
# =============================
# ============================================================
#  FINAL INPUT UI CONNECTED TO TRAINED MODEL
# ============================================================

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import numpy as np

display(HTML("""
<h2 style='color:#a78bfa;text-align:center'>
🎯 STUDENT RESULT PREDICTION (FINAL SYSTEM)
</h2>
"""))

# =============================
#  STUDENT DETAILS
# =============================
name = widgets.Text(description="Name:", layout=widgets.Layout(width='400px'))
usn  = widgets.Text(description="USN:", layout=widgets.Layout(width='400px'))

display(name, usn)

# =============================
# SEM 1 INPUT (USED FOR MODEL)
# =============================
display(HTML("<h3>📘 Semester 1 Marks</h3>"))

sem1_labels = [p[0] for p in pairs]

sem1_boxes = []
for label in sem1_labels:
    box = widgets.FloatText(
        description=label,
        layout=widgets.Layout(width='500px')
    )
    sem1_boxes.append(box)

display(widgets.VBox(sem1_boxes))

# =============================
#  SEM 2 INTERNALS
# =============================
display(HTML("<h3>📗 Semester 2 Internals</h3>"))

sem2_labels = [p[0] for p in pairs]

sem2_boxes = []
for label in sem2_labels:
    box = widgets.FloatText(
        description=label,
        layout=widgets.Layout(width='500px')
    )
    sem2_boxes.append(box)

display(widgets.VBox(sem2_boxes))

# =============================
#  BUTTON
# =============================
btn = widgets.Button(description="🚀 Predict Result", button_style='success')
out = widgets.Output()

def predict(b):
    with out:
        clear_output()

        # Prepare input
        sem1_vals = [x.value for x in sem1_boxes]
        sem2_vals = [x.value for x in sem2_boxes]

        full_input = np.array(sem1_vals + sem2_vals).reshape(1, -1)

        # Scale
        scaled = scaler.transform(full_input)

        # Predict using autoencoder
        recon = scaler.inverse_transform(model.predict(scaled))

        predicted_finals = recon[:, len(pairs):]

        total = 0

        print(f"\n👤 {name.value} ({usn.value})\n")

        for i, (internal_name, final_name) in enumerate(pairs):

            internal = sem2_vals[i]

            max_mark = 70 if i < 4 else 35
            final_pred = np.clip(predicted_finals[0][i], 0, max_mark)

            subject_total = internal + final_pred
            total += subject_total

            print(f"{internal_name}")
            print(f"  Internal: {internal}")
            print(f"  Predicted Final: {final_pred:.2f}")
            print(f"  Total: {subject_total:.2f}\n")

        # =============================
        #  FINAL RESULT
        # =============================
        cgpa = (total / 650) * 10

        def grade(c):
            if c >= 9: return "O"
            elif c >= 8: return "A+"
            elif c >= 7: return "A"
            elif c >= 6: return "B+"
            elif c >= 5: return "B"
            else: return "F"

        result = "Pass" if cgpa >= 5 else "Fail"

        display(HTML(f"""
        <div style="background:#111827;padding:20px;border-radius:10px;color:white">
            <h3>📊 FINAL RESULT</h3>
            <p>Total: {total:.2f} / 650</p>
            <p>CGPA: {cgpa:.2f}</p>
            <p>Grade: {grade(cgpa)}</p>
            <p>Result: {result}</p>
        </div>
        """))

btn.on_click(predict)

display(btn, out)

Text(value='', description='Name:', layout=Layout(width='400px'))

Text(value='', description='USN:', layout=Layout(width='400px'))

Button(button_style='success', description='🚀 Predict Result', style=ButtonStyle())

Output()